In [171]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler 


Up untill now I am just downloading the dataset, but in this dataset the index is multi-index so wi first flattened it

In [172]:
# now because csv dont parse the date time so this might be an issue for me, as the data will be trained in the future project based on 
# rolling window approach so it is really crucial to make sure that the data has the datetime sorted 

FinalStocks = pd.read_csv('data/raw_stocks.csv', parse_dates=["Date"])
# this is the first step to clean the data cleaning and making sure that the date column is of type datetime only 

In [173]:
FinalStocks[FinalStocks["Ticker"] == 'CBA.AX'].head()

,Date,Close,High,Low,Open,Volume,Ticker
4109,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX
4110,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX
4111,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX
4112,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX
4113,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX


In [174]:
print(FinalStocks['Ticker'].unique())

['BHP.AX' 'CBA.AX' 'CSL.AX' 'NAB.AX' 'WBC.AX' 'ANZ.AX' 'WES.AX' 'MQG.AX'
 'WOW.AX' 'TLS.AX']


In [175]:
CBA_Stock = FinalStocks[FinalStocks['Ticker'] == 'CBA.AX'].copy()

In [176]:
CBA_Stock = CBA_Stock.sort_values('Date').reset_index(drop=True)

In [177]:
print("Shape:", CBA_Stock.shape)

print("\nDate Range:")
print(CBA_Stock['Date'].min(), "→", CBA_Stock['Date'].max())

print("\nCheck ordering:")
print(CBA_Stock['Date'].is_monotonic_increasing)

print("\nMissing values:")
print(CBA_Stock.isnull().sum())

Shape: (4109, 7)

Date Range:
2010-01-04 00:00:00 → 2026-04-02 00:00:00

Check ordering:
True

Missing values:
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
Ticker    0
dtype: int64


In [178]:
CBA_Stock['Return'] = CBA_Stock['Close'].pct_change()

CBA_Stock['LogReturn'] = np.log(CBA_Stock['Close'] / CBA_Stock['Close'].shift(1))

In [179]:
CBA_Stock.head()

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn
0,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX,NaN,NaN
1,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX,0.015126,0.015013
2,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX,0.005027,0.005014
3,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX,-0.009646,-0.009693
4,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX,0.012987,0.012904


In [180]:
CBA_Stock['SMA_5'] = CBA_Stock['Close'].rolling(5).mean()
CBA_Stock['SMA_10'] = CBA_Stock['Close'].rolling(10).mean()

In [181]:
CBA_Stock.head(10)

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,SMA_10
0,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX,NaN,NaN,NaN,NaN
1,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX,0.015126,0.015013,NaN,NaN
2,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX,0.005027,0.005014,NaN,NaN
3,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX,-0.009646,-0.009693,NaN,NaN
4,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX,0.012987,0.012904,24.968361,NaN
5,2010-01-11,25.390263,25.484516,25.246637,25.269078,2751882,CBA.AX,0.007300,0.007274,25.120963,NaN
6,2010-01-12,25.390263,25.592236,25.291520,25.336403,3539936,CBA.AX,0.000000,0.000000,25.199060,NaN
7,2010-01-13,25.138912,25.354350,25.071589,25.255607,2761569,CBA.AX,-0.009899,-0.009949,25.201753,NaN
8,2010-01-14,25.489004,25.623652,25.269077,25.313959,3020355,CBA.AX,0.013926,0.013830,25.322937,NaN
9,2010-01-15,26.076965,26.076965,25.224190,25.493487,4801453,CBA.AX,0.023067,0.022805,25.497081,25.232721


In [182]:
CBA_Stock['Return_std_5'] = CBA_Stock['Return'].rolling(5).std()
CBA_Stock['Return_std_10'] = CBA_Stock['Return'].rolling(10).std()

In [183]:
CBA_Stock['Return_lag1'] = CBA_Stock['Return'].shift(1)
CBA_Stock['Return_lag2'] = CBA_Stock['Return'].shift(2)
CBA_Stock['Return_lag3'] = CBA_Stock['Return'].shift(3)
CBA_Stock['Return_lag5'] = CBA_Stock['Return'].shift(5)

In [184]:
CBA_Stock['Momentum_5'] = CBA_Stock['Return'].rolling(5).sum()
CBA_Stock['Momentum_10'] = CBA_Stock['Return'].rolling(10).sum()

In [185]:
CBA_Stock.head(10)

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,SMA_10,Return_std_5,Return_std_10,Return_lag1,Return_lag2,Return_lag3,Return_lag5,Momentum_5,Momentum_10
0,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX,0.015126,0.015013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX,0.005027,0.005014,NaN,NaN,NaN,NaN,0.015126,NaN,NaN,NaN,NaN,NaN
3,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX,-0.009646,-0.009693,NaN,NaN,NaN,NaN,0.005027,0.015126,NaN,NaN,NaN,NaN
4,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX,0.012987,0.012904,24.968361,NaN,NaN,NaN,-0.009646,0.005027,0.015126,NaN,NaN,NaN
5,2010-01-11,25.390263,25.484516,25.246637,25.269078,2751882,CBA.AX,0.007300,0.007274,25.120963,NaN,0.009739,NaN,0.012987,-0.009646,0.005027,NaN,0.030795,NaN
6,2010-01-12,25.390263,25.592236,25.291520,25.336403,3539936,CBA.AX,0.000000,0.000000,25.199060,NaN,0.008532,NaN,0.007300,0.012987,-0.009646,0.015126,0.015668,NaN
7,2010-01-13,25.138912,25.354350,25.071589,25.255607,2761569,CBA.AX,-0.009899,-0.009949,25.201753,NaN,0.010160,NaN,0.000000,0.007300,0.012987,0.005027,0.000742,NaN
8,2010-01-14,25.489004,25.623652,25.269077,25.313959,3020355,CBA.AX,0.013926,0.013830,25.322937,NaN,0.009946,NaN,-0.009899,0.000000,0.007300,-0.009646,0.024314,NaN
9,2010-01-15,26.076965,26.076965,25.224190,25.493487,4801453,CBA.AX,0.023067,0.022805,25.497081,25.232721,0.012656,NaN,0.013926,-0.009899,0.000000,0.012987,0.034395,NaN


In [186]:
CBA_Stock['Close_SMA5_diff'] = CBA_Stock['Close'] - CBA_Stock['SMA_5']
CBA_Stock['Close_SMA10_diff'] = CBA_Stock['Close'] - CBA_Stock['SMA_10']

In [187]:
CBA_Stock['TargetReturn'] = CBA_Stock['Return'].shift(-1)
CBA_Stock['TargetTrend'] = (CBA_Stock['TargetReturn'] > 0).astype(int)

In [188]:
CBA_Stock.head()

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,...,Return_lag1,Return_lag2,Return_lag3,Return_lag5,Momentum_5,Momentum_10,Close_SMA5_diff,Close_SMA10_diff,TargetReturn,TargetTrend
0,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015126,1
1,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX,0.015126,0.015013,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.005027,1
2,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX,0.005027,0.005014,NaN,...,0.015126,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.009646,0
3,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX,-0.009646,-0.009693,NaN,...,0.005027,0.015126,NaN,NaN,NaN,NaN,NaN,NaN,0.012987,1
4,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX,0.012987,0.012904,24.968361,...,-0.009646,0.005027,0.015126,NaN,NaN,NaN,0.237885,NaN,0.007300,1


In [189]:
CBA_Stock = CBA_Stock.dropna().reset_index(drop=True)

In [190]:
print(CBA_Stock['TargetTrend'].value_counts(normalize=True))

TargetTrend
1    0.526354
0    0.473646
Name: proportion, dtype: float64


In [191]:
CBA_Stock.head(10)

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,...,Return_lag1,Return_lag2,Return_lag3,Return_lag5,Momentum_5,Momentum_10,Close_SMA5_diff,Close_SMA10_diff,TargetReturn,TargetTrend
0,2010-01-18,26.054525,26.323823,25.816647,26.054525,6117434,CBA.AX,-0.000861,-0.000861,25.629934,...,0.023067,0.013926,-0.009899,0.007300,0.026234,0.057028,0.424591,0.679077,-0.024289,0
1,2010-01-19,25.421679,26.027597,25.381284,26.014132,5840218,CBA.AX,-0.024289,-0.024589,25.636217,...,-0.000861,0.023067,0.013926,0.000000,0.001944,0.017613,-0.214539,0.004040,-0.001412,0
2,2010-01-20,25.385778,25.722399,25.278059,25.717912,3761697,CBA.AX,-0.001412,-0.001413,25.685590,...,-0.024289,-0.000861,0.023067,-0.009899,0.010432,0.011174,-0.299812,-0.057893,0.000000,0
3,2010-01-21,25.385778,25.699959,25.273571,25.381289,5142694,CBA.AX,0.000000,0.000000,25.664945,...,-0.001412,-0.024289,-0.000861,0.013926,-0.003495,0.020820,-0.279167,-0.108163,-0.014321,0
4,2010-01-22,25.022221,25.085056,24.690087,24.910013,5067217,CBA.AX,-0.014321,-0.014425,25.453996,...,0.000000,-0.001412,-0.024289,0.023067,-0.040883,-0.006489,-0.431776,-0.453318,-0.010762,0
5,2010-01-25,24.752924,24.842689,24.577881,24.685599,3028846,CBA.AX,-0.010762,-0.010821,25.193676,...,-0.014321,0.000000,-0.001412,-0.000861,-0.050785,-0.024552,-0.440752,-0.658881,-0.009429,0
6,2010-01-27,24.519535,24.645208,24.326538,24.604813,4035586,CBA.AX,-0.009429,-0.009473,25.013247,...,-0.010762,-0.014321,0.000000,-0.024289,-0.035925,-0.033980,-0.493712,-0.805197,0.008603,1
7,2010-01-28,24.730480,24.775363,24.550948,24.721503,5277686,CBA.AX,0.008603,0.008566,24.882188,...,-0.009429,-0.010762,-0.014321,-0.001412,-0.025909,-0.015478,-0.151707,-0.553409,-0.033938,0
8,2010-01-29,23.891169,24.434252,23.787938,24.371415,9940798,CBA.AX,-0.033938,-0.034528,24.583266,...,0.008603,-0.009429,-0.010762,0.000000,-0.059848,-0.063342,-0.692097,-1.232937,-0.004697,0
9,2010-02-01,23.778961,24.120072,23.778961,23.877704,4273994,CBA.AX,-0.004697,-0.004708,24.334614,...,-0.033938,0.008603,-0.009429,-0.014321,-0.050223,-0.091106,-0.555653,-1.115344,0.010759,1


In [192]:
CBA_Stock.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4098 entries, 0 to 4097
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              4098 non-null   datetime64[ns]
 1   Close             4098 non-null   float64       
 2   High              4098 non-null   float64       
 3   Low               4098 non-null   float64       
 4   Open              4098 non-null   float64       
 5   Volume            4098 non-null   int64         
 6   Ticker            4098 non-null   object        
 7   Return            4098 non-null   float64       
 8   LogReturn         4098 non-null   float64       
 9   SMA_5             4098 non-null   float64       
 10  SMA_10            4098 non-null   float64       
 11  Return_std_5      4098 non-null   float64       
 12  Return_std_10     4098 non-null   float64       
 13  Return_lag1       4098 non-null   float64       
 14  Return_lag2       4098 n

In [193]:
feature_columns = [
    'Return',
    'Volume',

    'Return_lag1',
    'Return_lag2',
    'Return_lag3',
    'Return_lag5',

    'Momentum_5',
    'Momentum_10',

    'Return_std_5',
    'Return_std_10',

    'Close_SMA5_diff',
    'Close_SMA10_diff'
]

In [194]:
TrainingEnd = '2021-12-31'
ValidationEnd = '2023-12-31'

Train = CBA_Stock[CBA_Stock['Date'] <= TrainingEnd]
Validation = CBA_Stock[(CBA_Stock['Date'] > TrainingEnd) & (CBA_Stock['Date'] <= ValidationEnd)]
Test = CBA_Stock[CBA_Stock['Date'] > ValidationEnd]

In [195]:
X_train = Train[feature_columns]
X_val = Validation[feature_columns]
X_test = Test[feature_columns]

y_train = Train['TargetTrend']
y_train_reg = Train['TargetReturn']
y_val = Validation['TargetTrend']
y_test = Test['TargetTrend']

In [196]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [197]:
from sklearn.linear_model import LogisticRegression

In [198]:
LRModel = LogisticRegression(max_iter=1000, random_state=42)
LRModel.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [199]:
y_val_pred = LRModel.predict(X_val_scaled)
y_val_prob = LRModel.predict_proba(X_val_scaled)[:, 1]

y_test_pred = LRModel.predict(X_test_scaled)
y_test_prob = LRModel.predict_proba(X_test_scaled)[:, 1]

In [200]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

def ClassificationEvaluation(model_name, y_true, y_pred, y_prob):
    print(model_name)
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1 Score : {f1_score(y_true, y_pred):.4f}")
    print(f"AUC      : {roc_auc_score(y_true, y_prob):.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

In [201]:
ClassificationEvaluation(
    "Logistic Regression - Validation",
    y_val,
    y_val_pred,
    y_val_prob
)
ClassificationEvaluation(
    "Logistic Regression - Test",
    y_test,
    y_test_pred,
    y_test_prob
)

Logistic Regression - Validation
Accuracy : 0.5129
F1 Score : 0.6293
AUC      : 0.5065

Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.21      0.29       237
           1       0.53      0.78      0.63       266

    accuracy                           0.51       503
   macro avg       0.49      0.50      0.46       503
weighted avg       0.50      0.51      0.47       503

Confusion Matrix:
[[ 50 187]
 [ 58 208]]
Logistic Regression - Test
Accuracy : 0.5035
F1 Score : 0.6075
AUC      : 0.4551

Classification Report:
              precision    recall  f1-score   support

           0       0.40      0.28      0.32       247
           1       0.55      0.68      0.61       323

    accuracy                           0.50       570
   macro avg       0.47      0.48      0.47       570
weighted avg       0.48      0.50      0.48       570

Confusion Matrix:
[[ 68 179]
 [104 219]]


In [202]:
coef_df = pd.DataFrame({
    'Feature': feature_columns,
    'Coefficient': LRModel.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print(coef_df)

             Feature  Coefficient
11  Close_SMA10_diff     0.293344
2        Return_lag1     0.136874
0             Return     0.125660
3        Return_lag2     0.049394
4        Return_lag3     0.009804
1             Volume     0.005517
8       Return_std_5     0.003589
5        Return_lag5    -0.022388
9      Return_std_10    -0.041203
7        Momentum_10    -0.050345
6         Momentum_5    -0.197610
10   Close_SMA5_diff    -0.259841


In [203]:
from xgboost import XGBClassifier

XBGclass = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

XBGclass.fit(X_train, y_train)

y_val_pred = XBGclass.predict(X_val)
y_val_prob = XBGclass.predict_proba(X_val)[:, 1]

y_test_pred = XBGclass.predict(X_test)
y_test_prob = XBGclass.predict_proba(X_test)[:, 1]

In [204]:
ClassificationEvaluation(
    "XGB - Validation",
    y_val,
    y_val_pred,
    y_val_prob
)
ClassificationEvaluation(
    "XGB - Test",
    y_test,
    y_test_pred,
    y_test_prob
)

XGB - Validation
Accuracy : 0.5129
F1 Score : 0.5694
AUC      : 0.4909

Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.41      0.44       237
           1       0.53      0.61      0.57       266

    accuracy                           0.51       503
   macro avg       0.51      0.51      0.50       503
weighted avg       0.51      0.51      0.51       503

Confusion Matrix:
[[ 96 141]
 [104 162]]
XGB - Test
Accuracy : 0.5228
F1 Score : 0.5976
AUC      : 0.5221

Classification Report:
              precision    recall  f1-score   support

           0       0.44      0.39      0.41       247
           1       0.57      0.63      0.60       323

    accuracy                           0.52       570
   macro avg       0.51      0.51      0.51       570
weighted avg       0.52      0.52      0.52       570

Confusion Matrix:
[[ 96 151]
 [121 202]]


Moving Forward to new sentiment first using FinBERT

In [205]:
import json
import pandas as pd

file_path = "data/r_AusFinance_posts-2012-24.jsonl"  # change if needed

rows = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            post = json.loads(line)

            created = pd.to_datetime(post.get("created_utc"), unit="s", utc=True)

            title = post.get("title", "")
            selftext = post.get("selftext", "")

            rows.append({
                "datetime": created,
                "date": created.date(),
                "subreddit": post.get("subreddit"),
                "title": title,
                "selftext": selftext,
                "text": f"{title} {selftext}".strip(),
                "score": post.get("score"),
                "num_comments": post.get("num_comments"),
                "flair": post.get("link_flair_text"),
                "url": post.get("url"),
                "permalink": post.get("permalink")
            })

reddit_df = pd.DataFrame(rows)

reddit_df = reddit_df[
    reddit_df["text"].notna()
    & (reddit_df["text"].str.strip() != "")
    & (~reddit_df["text"].str.lower().isin(["[removed]", "[deleted]"]))
].reset_index(drop=True)

print("Shape:", reddit_df.shape)
print("Date range:", reddit_df["date"].min(), "→", reddit_df["date"].max())
print("Columns:", reddit_df.columns.tolist())
display(reddit_df.head())

Shape: (135213, 11)
Date range: 2021-01-01 → 2024-12-30
Columns: ['datetime', 'date', 'subreddit', 'title', 'selftext', 'text', 'score', 'num_comments', 'flair', 'url', 'permalink']


,datetime,date,subreddit,title,selftext,text,score,num_comments,flair,url,permalink
0,2021-01-01 00:47:32+00:00,2021-01-01,AusFinance,Will commsec record all of my trades for tax t...,[deleted],Will commsec record all of my trades for tax t...,78,37,Tax,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/ko1sr1/will_commsec_rec...
1,2021-01-01 01:16:58+00:00,2021-01-01,AusFinance,How do i invest in the new Western Sydney airp...,[removed],How do i invest in the new Western Sydney airp...,1,1,None,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/ko29mb/how_do_i_invest_...
2,2021-01-01 01:54:58+00:00,2021-01-01,AusFinance,Will having USD in a foreign trading account a...,My partner and I are saving for our first home...,Will having USD in a foreign trading account a...,7,8,Lifestyle,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/ko2u3z/will_having_usd_...
3,2021-01-01 02:47:31+00:00,2021-01-01,AusFinance,Instantaneous Payment on a Weekend.,"Hey Folks, is there a way to do an instantaneo...","Instantaneous Payment on a Weekend. Hey Folks,...",1,5,None,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/ko3lf5/instantaneous_pa...
4,2021-01-01 02:53:44+00:00,2021-01-01,AusFinance,Income and Spending Results for 2020,[deleted],Income and Spending Results for 2020 [deleted],0,2,None,https://i.redd.it/oovp85rnwm861.png,/r/AusFinance/comments/ko3ois/income_and_spend...


In [206]:
reddit_df["title"] = reddit_df["title"].replace(["[deleted]", "[removed]"], "")
reddit_df["selftext"] = reddit_df["selftext"].replace(["[deleted]", "[removed]"], "")

reddit_df["text"] = (
    reddit_df["title"].fillna("") + " " + reddit_df["selftext"].fillna("")
).str.strip()

reddit_df = reddit_df[
    reddit_df["text"].notna() &
    (reddit_df["text"].str.strip() != "")
].reset_index(drop=True)

print("Cleaned shape:", reddit_df.shape)
display(reddit_df.head())

Cleaned shape: (135213, 11)


,datetime,date,subreddit,title,selftext,text,score,num_comments,flair,url,permalink
0,2021-01-01 00:47:32+00:00,2021-01-01,AusFinance,Will commsec record all of my trades for tax t...,,Will commsec record all of my trades for tax t...,78,37,Tax,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/ko1sr1/will_commsec_rec...
1,2021-01-01 01:16:58+00:00,2021-01-01,AusFinance,How do i invest in the new Western Sydney airp...,,How do i invest in the new Western Sydney airp...,1,1,None,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/ko29mb/how_do_i_invest_...
2,2021-01-01 01:54:58+00:00,2021-01-01,AusFinance,Will having USD in a foreign trading account a...,My partner and I are saving for our first home...,Will having USD in a foreign trading account a...,7,8,Lifestyle,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/ko2u3z/will_having_usd_...
3,2021-01-01 02:47:31+00:00,2021-01-01,AusFinance,Instantaneous Payment on a Weekend.,"Hey Folks, is there a way to do an instantaneo...","Instantaneous Payment on a Weekend. Hey Folks,...",1,5,None,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/ko3lf5/instantaneous_pa...
4,2021-01-01 02:53:44+00:00,2021-01-01,AusFinance,Income and Spending Results for 2020,,Income and Spending Results for 2020,0,2,None,https://i.redd.it/oovp85rnwm861.png,/r/AusFinance/comments/ko3ois/income_and_spend...


In [207]:
keywords_strict = [
    "asx",
    "stock",
    "stocks",
    "share",
    "shares",
    "trading",
    "market",
    "portfolio",
    "dividend",
    "earnings",
    "bull",
    "bear",
    "buy",
    "sell",
    "valuation",
    "cba",
    "commonwealth bank",
    "rba",
    "interest rate",
    "inflation"
]

In [208]:
exclude_keywords = [
    "property",
    "mortgage",
    "house",
    "rent",
    "salary",
    "job",
    "career",
    "budget",
    "saving",
    "superannuation",
    "tax return",
    "child",
    "family",
    "loan",
    "debt"
]

In [209]:
import re

include_pattern = re.compile("|".join(keywords_strict), re.IGNORECASE)
exclude_pattern = re.compile("|".join(exclude_keywords), re.IGNORECASE)

filtered_final = reddit_df[
    reddit_df["text"].str.contains(include_pattern, na=False) &
    ~reddit_df["text"].str.contains(exclude_pattern, na=False)
].reset_index(drop=True)

print("Final count:", len(filtered_final))
display(filtered_final.head(10))

Final count: 16283


,datetime,date,subreddit,title,selftext,text,score,num_comments,flair,url,permalink
0,2021-01-01 03:08:42+00:00,2021-01-01,AusFinance,Westpac negative interest charge,I have got a negative charge in my westpac cho...,Westpac negative interest charge I have got a ...,11,6,None,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/ko3w4d/westpac_negative...
1,2021-01-01 04:27:30+00:00,2021-01-01,AusFinance,Initiating trades,I have recently come into some decent capital ...,Initiating trades I have recently come into so...,5,7,None,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/ko4ya2/initiating_trades/
2,2021-01-01 06:16:02+00:00,2021-01-01,AusFinance,"DSSP, what's the catch?","Hi all, I'm a top rate tax payer looking to in...","DSSP, what's the catch? Hi all, I'm a top rate...",5,10,None,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/ko6g01/dssp_whats_the_c...
3,2021-01-01 12:12:48+00:00,2021-01-01,AusFinance,How does the dropping USD affect valuations in...,I was reading an article how the USD is gettin...,How does the dropping USD affect valuations in...,3,5,Investing,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/koacq0/how_does_the_dro...
4,2021-01-01 13:02:26+00:00,2021-01-01,AusFinance,Ways to profit from a drop in share price?,If I believe the market or a share is going to...,Ways to profit from a drop in share price? If ...,0,13,Investing,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/koayit/ways_to_profit_f...
5,2021-01-02 04:18:02+00:00,2021-01-02,AusFinance,ASX index value,"Hi All,\n\nI am aware of how to get a share va...","ASX index value Hi All,\n\nI am aware of how t...",1,8,Investing,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/koql7q/asx_index_value/
6,2021-01-02 05:47:54+00:00,2021-01-02,AusFinance,Business Bank Alternates,"Hi,\n\nI'm researching a number of everyday bu...","Business Bank Alternates Hi,\n\nI'm researchin...",7,8,Business,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/korxg3/business_bank_al...
7,2021-01-02 06:14:16+00:00,2021-01-02,AusFinance,New ASX website- absolutley useless,The new ASX website is absolutley useless for ...,New ASX website- absolutley useless The new AS...,68,24,Investing,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/kosazc/new_asx_website_...
8,2021-01-02 07:12:07+00:00,2021-01-02,AusFinance,Employer Share Plan Question,,Employer Share Plan Question,1,2,Investing,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/kot2go/employer_share_p...
9,2021-01-02 08:22:48+00:00,2021-01-02,AusFinance,Living above a supermarket/grocery store,"Hi,\n\nI am looking at a 2 bedroom 2 bathroom ...","Living above a supermarket/grocery store Hi,\n...",0,29,None,https://www.reddit.com/r/AusFinance/comments/k...,/r/AusFinance/comments/kotx3v/living_above_a_s...


In [210]:
print("Filtered count:", len(filtered_final))
print("Date range:", filtered_final["date"].min(), "→", filtered_final["date"].max())

display(filtered_final[["date", "title", "selftext", "text", "score", "num_comments", "flair"]].sample(10, random_state=42))

Filtered count: 16283
Date range: 2021-01-01 → 2024-12-30


,date,title,selftext,text,score,num_comments,flair
11959,2023-12-07,In Case You Missed It: A lithium project acqui...,\n\nhttps://preview.redd.it/6hadgf4omy4c1.png...,In Case You Missed It: A lithium project acqui...,1,2,None
15161,2024-09-13,[Tourist Refund Scheme] VAT refund question fo...,"Hi All, \n\nSmall conundrum for anyone well ve...",[Tourist Refund Scheme] VAT refund question fo...,0,6,None
1655,2021-05-15,Total noob looking for advice,For reference I’m 24 with about 180k to my nam...,Total noob looking for advice For reference I’...,0,8,Lifestyle
11114,2023-09-22,Why is the stock market falling so much?,,Why is the stock market falling so much?,1,0,Investing
13663,2024-05-05,The first signs.,"Ok, what are the first signs that the housing ...","The first signs. Ok, what are the first signs ...",0,52,None
6057,2022-08-04,Inflation isn't the 6.1% they say it is – for ...,Interesting analysis. Never bothered to figure...,Inflation isn't the 6.1% they say it is – for ...,0,25,Business
15845,2024-11-14,Advice on redraw/offset. First home buyer.,,Advice on redraw/offset. First home buyer.,1,0,None
14205,2024-06-26,Free Email Alerts for ASX Announcements,,Free Email Alerts for ASX Announcements,1,0,None
1071,2021-03-20,Time to increase interest rates!,,Time to increase interest rates!,0,2,None
13477,2024-04-18,Help - have made a grave error with computershare,I had bought some shares in 2020 when the cost...,Help - have made a grave error with computersh...,23,10,None


In [211]:
import torch
import numpy as np
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
finbert_model.to(device)

labels = ["positive", "negative", "neutral"]

def run_finbert_batch(texts, batch_size=16, max_length=128):
    all_results = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size].tolist()

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = finbert_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()

        for p in probs:
            pred_idx = np.argmax(p)
            all_results.append({
                "finbert_label": labels[pred_idx],
                "finbert_positive": p[0],
                "finbert_negative": p[1],
                "finbert_neutral": p[2],
                "sentiment_score": p[0] - p[1]
            })

    return pd.DataFrame(all_results)

sentiment_results = run_finbert_batch(filtered_final["text"], batch_size=16)

filtered_sentiment = pd.concat(
    [filtered_final.reset_index(drop=True), sentiment_results],
    axis=1
)

print(filtered_sentiment.shape)
display(filtered_sentiment[[
    "date", "text", "finbert_label",
    "finbert_positive", "finbert_negative",
    "finbert_neutral", "sentiment_score"
]].head())

100%|██████████| 1018/1018 [05:59<00:00,  2.83it/s]


(16283, 16)


,date,text,finbert_label,finbert_positive,finbert_negative,finbert_neutral,sentiment_score
0,2021-01-01,Westpac negative interest charge I have got a ...,negative,0.020747,0.826863,0.152390,-0.806116
1,2021-01-01,Initiating trades I have recently come into so...,neutral,0.080050,0.015444,0.904506,0.064606
2,2021-01-01,"DSSP, what's the catch? Hi all, I'm a top rate...",neutral,0.072065,0.022131,0.905804,0.049933
3,2021-01-01,How does the dropping USD affect valuations in...,neutral,0.046563,0.433323,0.520113,-0.386760
4,2021-01-01,Ways to profit from a drop in share price? If ...,negative,0.028850,0.536546,0.434604,-0.507696


In [212]:
print(filtered_sentiment["finbert_label"].value_counts())
print(filtered_sentiment["sentiment_score"].describe())

finbert_label
neutral     14028
negative     1706
positive      549
Name: count, dtype: int64
count    16283.000000
mean        -0.050258
std          0.287254
min         -0.967517
25%         -0.057543
50%          0.007866
75%          0.050496
max          0.937288
Name: sentiment_score, dtype: float64


In [213]:
daily_sentiment = (
    filtered_sentiment
    .groupby("date")
    .agg(
        reddit_post_count=("text", "count"),
        sentiment_mean=("sentiment_score", "mean"),
        sentiment_median=("sentiment_score", "median"),
        sentiment_std=("sentiment_score", "std"),
        sentiment_positive_mean=("finbert_positive", "mean"),
        sentiment_negative_mean=("finbert_negative", "mean"),
        sentiment_neutral_mean=("finbert_neutral", "mean"),
        avg_score=("score", "mean"),
        avg_comments=("num_comments", "mean")
    )
    .reset_index()
)

daily_sentiment["sentiment_std"] = daily_sentiment["sentiment_std"].fillna(0)

print("Daily sentiment shape:", daily_sentiment.shape)
print("Date range:", daily_sentiment["date"].min(), "→", daily_sentiment["date"].max())
display(daily_sentiment.head())
display(daily_sentiment.describe())

Daily sentiment shape: (1460, 10)
Date range: 2021-01-01 → 2024-12-30


,date,reddit_post_count,sentiment_mean,sentiment_median,sentiment_std,sentiment_positive_mean,sentiment_negative_mean,sentiment_neutral_mean,avg_score,avg_comments
0,2021-01-01,5,-0.317206,-0.386760,0.374409,0.049655,0.366861,0.583484,4.800000,8.200000
1,2021-01-02,10,-0.051797,0.031009,0.183712,0.051762,0.103559,0.844680,9.500000,12.200000
2,2021-01-03,13,-0.066984,0.018300,0.240375,0.057991,0.124976,0.817033,8.769231,16.230769
3,2021-01-04,7,-0.073292,-0.048291,0.102454,0.048422,0.121714,0.829864,26.571429,24.714286
4,2021-01-05,15,-0.091173,0.009663,0.354827,0.071170,0.162343,0.766487,10.933333,6.866667


,reddit_post_count,sentiment_mean,sentiment_median,sentiment_std,sentiment_positive_mean,sentiment_negative_mean,sentiment_neutral_mean,avg_score,avg_comments
count,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,11.152740,-0.049887,-0.000566,0.250318,0.088335,0.138222,0.773443,21.435363,24.787646
std,4.800827,0.100606,0.059771,0.126317,0.046578,0.084476,0.092141,30.772487,24.114837
min,1.000000,-0.486822,-0.604617,0.000000,0.027147,0.009184,0.233511,0.000000,0.000000
25%,8.000000,-0.107139,-0.010026,0.144080,0.056431,0.071412,0.719403,4.305804,9.666667
50%,11.000000,-0.034315,0.008401,0.261438,0.074425,0.122913,0.785005,11.322917,17.333333
75%,14.000000,0.014281,0.023815,0.342741,0.106150,0.189310,0.843584,26.595238,31.200000
max,44.000000,0.400928,0.286671,0.858753,0.429629,0.551085,0.929167,328.000000,218.700000


In [214]:
sentiment_features = daily_sentiment.copy()

sentiment_features["date"] = pd.to_datetime(sentiment_features["date"])

sentiment_features = sentiment_features.sort_values("date")

# SHIFT → very important (prevents leakage)
sentiment_features.iloc[:, 1:] = sentiment_features.iloc[:, 1:].shift(1)

sentiment_features = sentiment_features.dropna().reset_index(drop=True)

print("After shift shape:", sentiment_features.shape)
display(sentiment_features.head())

After shift shape: (1459, 10)


/var/folders/5l/hdkvhlj11x73wsv279bv3wfh0000gn/T/ipykernel_54386/789600090.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0        NaN
1        5.0
2       10.0
3       13.0
4        7.0
        ... 
1455     6.0
1456     8.0
1457    12.0
1458     6.0
1459     6.0
Name: reddit_post_count, Length: 1460, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  sentiment_features.iloc[:, 1:] = sentiment_features.iloc[:, 1:].shift(1)


,date,reddit_post_count,sentiment_mean,sentiment_median,sentiment_std,sentiment_positive_mean,sentiment_negative_mean,sentiment_neutral_mean,avg_score,avg_comments
0,2021-01-02,5.0,-0.317206,-0.386760,0.374409,0.049655,0.366861,0.583484,4.800000,8.200000
1,2021-01-03,10.0,-0.051797,0.031009,0.183712,0.051762,0.103559,0.844680,9.500000,12.200000
2,2021-01-04,13.0,-0.066984,0.018300,0.240375,0.057991,0.124976,0.817033,8.769231,16.230769
3,2021-01-05,7.0,-0.073292,-0.048291,0.102454,0.048422,0.121714,0.829864,26.571429,24.714286
4,2021-01-06,15.0,-0.091173,0.009663,0.354827,0.071170,0.162343,0.766487,10.933333,6.866667


In [215]:
print(CBA_Stock.shape)
print(CBA_Stock.columns.tolist())
print(CBA_Stock["Date"].min(), "→", CBA_Stock["Date"].max())
display(CBA_Stock.head())
display(CBA_Stock.tail())

(4098, 23)
['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'Return', 'LogReturn', 'SMA_5', 'SMA_10', 'Return_std_5', 'Return_std_10', 'Return_lag1', 'Return_lag2', 'Return_lag3', 'Return_lag5', 'Momentum_5', 'Momentum_10', 'Close_SMA5_diff', 'Close_SMA10_diff', 'TargetReturn', 'TargetTrend']
2010-01-18 00:00:00 → 2026-04-01 00:00:00


,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,...,Return_lag1,Return_lag2,Return_lag3,Return_lag5,Momentum_5,Momentum_10,Close_SMA5_diff,Close_SMA10_diff,TargetReturn,TargetTrend
0,2010-01-18,26.054525,26.323823,25.816647,26.054525,6117434,CBA.AX,-0.000861,-0.000861,25.629934,...,0.023067,0.013926,-0.009899,0.007300,0.026234,0.057028,0.424591,0.679077,-0.024289,0
1,2010-01-19,25.421679,26.027597,25.381284,26.014132,5840218,CBA.AX,-0.024289,-0.024589,25.636217,...,-0.000861,0.023067,0.013926,0.000000,0.001944,0.017613,-0.214539,0.004040,-0.001412,0
2,2010-01-20,25.385778,25.722399,25.278059,25.717912,3761697,CBA.AX,-0.001412,-0.001413,25.685590,...,-0.024289,-0.000861,0.023067,-0.009899,0.010432,0.011174,-0.299812,-0.057893,0.000000,0
3,2010-01-21,25.385778,25.699959,25.273571,25.381289,5142694,CBA.AX,0.000000,0.000000,25.664945,...,-0.001412,-0.024289,-0.000861,0.013926,-0.003495,0.020820,-0.279167,-0.108163,-0.014321,0
4,2010-01-22,25.022221,25.085056,24.690087,24.910013,5067217,CBA.AX,-0.014321,-0.014425,25.453996,...,0.000000,-0.001412,-0.024289,0.023067,-0.040883,-0.006489,-0.431776,-0.453318,-0.010762,0


,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,...,Return_lag1,Return_lag2,Return_lag3,Return_lag5,Momentum_5,Momentum_10,Close_SMA5_diff,Close_SMA10_diff,TargetReturn,TargetTrend
4093,2026-03-26,173.179993,174.600006,171.940002,172.089996,1412725,CBA.AX,0.005866,0.005849,173.271997,...,0.006136,-0.017963,-0.007914,0.001525,-0.023572,0.009595,-0.092004,-1.442004,0.002599,1
4094,2026-03-27,173.630005,174.289993,172.169998,172.630005,1482825,CBA.AX,0.002599,0.002595,172.869998,...,0.005866,0.006136,-0.017963,-0.009698,-0.011276,-0.000394,0.760007,-0.978993,-0.028279,0
4095,2026-03-30,168.720001,172.539993,167.309998,172.000000,1592667,CBA.AX,-0.028279,-0.028686,171.763998,...,0.002599,0.005866,0.006136,-0.007914,-0.031640,-0.038859,-3.043997,-5.207997,-0.006046,0
4096,2026-03-31,167.699997,170.770004,167.070007,169.539993,2563551,CBA.AX,-0.006046,-0.006064,171.079999,...,-0.028279,0.002599,0.005866,-0.017963,-0.019723,-0.048265,-3.380002,-5.386002,0.025045,1
4097,2026-04-01,171.899994,171.899994,168.119995,168.940002,2275008,CBA.AX,0.025045,0.024736,171.025998,...,-0.006046,-0.028279,0.002599,0.006136,-0.000815,-0.028728,0.873996,-0.667004,0.005236,1


In [239]:
CBA_Stock["date"] = pd.to_datetime(CBA_Stock["Date"]).dt.normalize()
sentiment_features["date"] = pd.to_datetime(sentiment_features["date"]).dt.normalize()

merged_df = pd.merge(
    CBA_Stock,
    sentiment_features,
    on="date",
    how="left"
)

print("Merged shape:", merged_df.shape)
print("Missing sentiment rows:", merged_df["sentiment_mean"].isna().sum())

display(merged_df.head())
display(merged_df.tail())

Merged shape: (4098, 33)
Missing sentiment rows: 3088


,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,...,date,reddit_post_count,sentiment_mean,sentiment_median,sentiment_std,sentiment_positive_mean,sentiment_negative_mean,sentiment_neutral_mean,avg_score,avg_comments
0,2010-01-18,26.054525,26.323823,25.816647,26.054525,6117434,CBA.AX,-0.000861,-0.000861,25.629934,...,2010-01-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-19,25.421679,26.027597,25.381284,26.014132,5840218,CBA.AX,-0.024289,-0.024589,25.636217,...,2010-01-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-20,25.385778,25.722399,25.278059,25.717912,3761697,CBA.AX,-0.001412,-0.001413,25.685590,...,2010-01-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-01-21,25.385778,25.699959,25.273571,25.381289,5142694,CBA.AX,0.000000,0.000000,25.664945,...,2010-01-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-01-22,25.022221,25.085056,24.690087,24.910013,5067217,CBA.AX,-0.014321,-0.014425,25.453996,...,2010-01-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,...,date,reddit_post_count,sentiment_mean,sentiment_median,sentiment_std,sentiment_positive_mean,sentiment_negative_mean,sentiment_neutral_mean,avg_score,avg_comments
4093,2026-03-26,173.179993,174.600006,171.940002,172.089996,1412725,CBA.AX,0.005866,0.005849,173.271997,...,2026-03-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4094,2026-03-27,173.630005,174.289993,172.169998,172.630005,1482825,CBA.AX,0.002599,0.002595,172.869998,...,2026-03-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4095,2026-03-30,168.720001,172.539993,167.309998,172.000000,1592667,CBA.AX,-0.028279,-0.028686,171.763998,...,2026-03-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4096,2026-03-31,167.699997,170.770004,167.070007,169.539993,2563551,CBA.AX,-0.006046,-0.006064,171.079999,...,2026-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4097,2026-04-01,171.899994,171.899994,168.119995,168.940002,2275008,CBA.AX,0.025045,0.024736,171.025998,...,2026-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [260]:
# ============================================================
# CLEAN MODELING BLOCK AFTER merged_df IS CREATED
# ============================================================

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from xgboost import XGBClassifier

# 1. Final overlap dataset
final_df = merged_df.dropna(subset=["sentiment_mean"]).copy()
final_df = final_df.sort_values("date").reset_index(drop=True)

# 2. Sentiment engineered features
final_df["sentiment_abs"] = final_df["sentiment_mean"].abs()
final_df["sentiment_vol_weighted"] = final_df["sentiment_mean"] * final_df["reddit_post_count"]
final_df["sentiment_disagreement"] = final_df["sentiment_std"]

# 3. Shock features
final_df["sentiment_z"] = (
    final_df["sentiment_mean"] - final_df["sentiment_mean"].rolling(20).mean()
) / final_df["sentiment_mean"].rolling(20).std()

final_df["attention_spike"] = (
    final_df["reddit_post_count"] / final_df["reddit_post_count"].rolling(20).mean()
)

final_df["extreme_sentiment"] = (final_df["sentiment_z"].abs() > 1.5).astype(int)

# Drop rows created by rolling window
final_df = final_df.dropna().reset_index(drop=True)

print("Final clean dataset:", final_df.shape)
print("Date range:", final_df["date"].min(), "→", final_df["date"].max())

# 4. Time split
train_df = final_df[final_df["date"] < "2023-01-01"].copy()
val_df   = final_df[(final_df["date"] >= "2023-01-01") & (final_df["date"] < "2024-01-01")].copy()
test_df  = final_df[final_df["date"] >= "2024-01-01"].copy()

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

# 5. Feature sets
tech_features = [
    "Return_lag1", "Return_lag2", "Return_lag3", "Return_lag5",
    "Momentum_5", "Momentum_10",
    "Return_std_5", "Return_std_10"
]

sentiment_features_model = [
    "sentiment_mean",
    "sentiment_abs",
    "sentiment_vol_weighted",
    "sentiment_disagreement",
    "sentiment_z",
    "attention_spike",
    "extreme_sentiment"
]

all_features = tech_features + sentiment_features_model

# 6. Build X/y
X_train = train_df[all_features]
X_val   = val_df[all_features]
X_test  = test_df[all_features]

y_train = train_df["TargetTrend"]
y_val   = val_df["TargetTrend"]
y_test  = test_df["TargetTrend"]

print("Feature shapes:", X_train.shape, X_val.shape, X_test.shape)
print("NaNs:", X_train.isna().sum().sum(), X_val.isna().sum().sum(), X_test.isna().sum().sum())

# 7. Logistic Regression with scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000, class_weight="balanced")
log_model.fit(X_train_scaled, y_train)

log_val_proba = log_model.predict_proba(X_val_scaled)[:, 1]
log_val_pred = log_model.predict(X_val_scaled)

print("\nLOGISTIC REGRESSION VALIDATION")
print("Accuracy:", accuracy_score(y_val, log_val_pred))
print("F1:", f1_score(y_val, log_val_pred))
print("AUC:", roc_auc_score(y_val, log_val_proba))

# 8. XGBoost using SAME clean X/y
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]
xgb_val_pred = xgb_model.predict(X_val)

print("\nXGBOOST VALIDATION")
print("Accuracy:", accuracy_score(y_val, xgb_val_pred))
print("F1:", f1_score(y_val, xgb_val_pred))
print("AUC:", roc_auc_score(y_val, xgb_val_proba))

Final clean dataset: (991, 39)
Date range: 2021-02-01 00:00:00 → 2024-12-30 00:00:00
Train: (486, 39)
Val: (252, 39)
Test: (253, 39)
Feature shapes: (486, 15) (252, 15) (253, 15)
NaNs: 0 0 0

LOGISTIC REGRESSION VALIDATION
Accuracy: 0.49206349206349204
F1: 0.5223880597014925
AUC: 0.4740082513487781

XGBOOST VALIDATION
Accuracy: 0.4642857142857143
F1: 0.532871972318339
AUC: 0.42748333862265947
